In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [17]:
thai_den = [
    "Mư nảu bản.",
    "Khảu kin lẩu.",
    "Mư pây học.",
    "Nảu mưa lắm.",
    "Khảu thích kin cơm."
]

references = [
    "Tôi về bản.",
    "Anh ấy ăn cơm.",
    "Tôi đi học.",
    "Hôm nay mưa rất to.",
    "Anh ấy thích ăn cơm."
]

In [2]:
predictions = [
    "Tôi về bản.",
    "Anh ấy ăn cơm.",
    "Tôi đi trường.",
    "Hôm nay mưa rất to.",
    "Anh ấy thích cơm."
]

In [5]:
def get_ngrams(tokens, n):
    result = []

    for i in range(len(tokens) - n + 1):
        result.append(tuple(tokens[i:i+n]))

    return result

sentence = ["Tôi", "đi", "học"]

print(get_ngrams(sentence, 1))
print(get_ngrams(sentence, 2))

[('Tôi',), ('đi',), ('học',)]
[('Tôi', 'đi'), ('đi', 'học')]


In [6]:
from collections import Counter

def count_ngrams(tokens, n):
    return Counter(get_ngrams(tokens, n))


sentence = ["Tôi", "đi", "học"]

print(count_ngrams(sentence, 1))
print(count_ngrams(sentence, 2))

Counter({('Tôi',): 1, ('đi',): 1, ('học',): 1})
Counter({('Tôi', 'đi'): 1, ('đi', 'học'): 1})


In [7]:
reference = "Tôi đi học"
prediction = "Tôi đi trường"

ref_tokens = reference.split()
pred_tokens = prediction.split()

print(ref_tokens)
print(pred_tokens)

['Tôi', 'đi', 'học']
['Tôi', 'đi', 'trường']


In [8]:
ref_1gram = count_ngrams(ref_tokens, 1)
pred_1gram = count_ngrams(pred_tokens, 1)

print(ref_1gram)
print(pred_1gram)

matched = 0

for gram, count in pred_1gram.items():
    matched += min(count, ref_1gram.get(gram, 0))

print(matched)

Counter({('Tôi',): 1, ('đi',): 1, ('học',): 1})
Counter({('Tôi',): 1, ('đi',): 1, ('trường',): 1})
2


In [9]:
def ngram_precision(reference, prediction, n):

    ref_tokens = reference.split()
    pred_tokens = prediction.split()

    ref_ngrams = count_ngrams(ref_tokens, n)
    pred_ngrams = count_ngrams(pred_tokens, n)

    matched = 0

    for gram, count in pred_ngrams.items():
        matched += min(count, ref_ngrams.get(gram, 0))

    total = sum(pred_ngrams.values())

    if total == 0:
        return 0

    return matched / total

reference = "Tôi đi học"
prediction = "Tôi đi trường"

print(ngram_precision(reference, prediction, 1))
print(ngram_precision(reference, prediction, 2))

0.6666666666666666
0.5


In [13]:
from collections import Counter

def count_ngrams(tokens, n):
    return Counter(get_ngrams(tokens, n))

In [14]:
def ngram_precision(reference, prediction, n):

    ref_tokens = reference.split()
    pred_tokens = prediction.split()

    ref_ngrams = count_ngrams(ref_tokens, n)
    pred_ngrams = count_ngrams(pred_tokens, n)

    matched = 0

    for gram, count in pred_ngrams.items():
        matched += min(count, ref_ngrams.get(gram, 0))

    total = sum(pred_ngrams.values())

    if total == 0:
        return 0

    return matched / total

In [15]:
import math

def bleu_score(references, predictions):

    precisions = []

    # Tính 1-gram, 2-gram, 3-gram, 4-gram
    for n in range(1, 5):

        total_matched = 0
        total_ngrams = 0

        # Duyệt toàn bộ dataset
        for reference, prediction in zip(references, predictions):

            ref_tokens = reference.split()
            pred_tokens = prediction.split()

            ref_ngrams = count_ngrams(ref_tokens, n)
            pred_ngrams = count_ngrams(pred_tokens, n)

            # Đếm n-gram khớp
            for gram, count in pred_ngrams.items():
                total_matched += min(
                    count,
                    ref_ngrams.get(gram, 0)
                )

            total_ngrams += sum(pred_ngrams.values())

        # Precision của n-gram
        if total_ngrams == 0:
            precision = 0
        else:
            precision = total_matched / total_ngrams

        precisions.append(precision)

    # Nếu có n-gram không khớp hoàn toàn
    if min(precisions) == 0:
        return 0

    # Trung bình hình học của 4 precision
    score = 0

    for p in precisions:
        score += math.log(p)

    score = math.exp(score / 4)

    # Brevity Penalty
    reference_length = sum(
        len(x.split()) for x in references
    )

    prediction_length = sum(
        len(x.split()) for x in predictions
    )

    if prediction_length == 0:
        return 0

    if prediction_length >= reference_length:
        bp = 1
    else:
        bp = math.exp(
            1 - reference_length / prediction_length
        )

    return bp * score

In [18]:
bleu = bleu_score(references, predictions)

print("BLEU =", bleu)
print("BLEU (%) =", bleu * 100)

BLEU = 0.7870722901826883
BLEU (%) = 78.70722901826883


In [8]:
from collections import Counter

# ==============================
# DATASET
# ==============================

thai_den = [
    "Mư nảu bản.",
    "Khảu kin lẩu.",
    "Mư pây học.",
    "Nảu mưa lắm.",
    "Khảu thích kin cơm."
]

references = [
    "Tôi về bản.",
    "Anh ấy ăn cơm.",
    "Tôi đi học.",
    "Hôm nay mưa rất to.",
    "Anh ấy thích ăn cơm."
]

predictions = [
    "Tôi về bản.",
    "Anh ấy ăn cơm.",
    "Tôi đi trường.",
    "Hôm nay mưa rất to.",
    "Anh ấy thích cơm."
]


# ==============================
# HÀM TẠO CHARACTER N-GRAM
# ==============================

def get_ngrams(text, n):
    return Counter(
        text[i:i+n]
        for i in range(len(text) - n + 1)
    )


# ==============================
# HÀM chrF CHO 1 CÂU
# ==============================

def chrf_score(prediction, reference, max_n=6, beta=2):

    precisions = []
    recalls = []

    for n in range(1, max_n + 1):

        pred = get_ngrams(prediction, n)
        ref = get_ngrams(reference, n)

        # số n-gram giống nhau
        common = sum((pred & ref).values())

        pred_count = sum(pred.values())
        ref_count = sum(ref.values())

        # Precision
        if pred_count > 0:
            precision = common / pred_count
        else:
            precision = 0

        # Recall
        if ref_count > 0:
            recall = common / ref_count
        else:
            recall = 0

        precisions.append(precision)
        recalls.append(recall)

    # Trung bình Precision và Recall
    precision = sum(precisions) / len(precisions)
    recall = sum(recalls) / len(recalls)

    # F-beta
    if precision == 0 and recall == 0:
        return 0

    score = (
        (1 + beta**2) * precision * recall
        / (beta**2 * precision + recall)
    )

    return score * 100


# ==============================
# TÍNH TỪNG CÂU
# ==============================

print("===== chrF TỪNG CÂU =====")

scores = []

for i in range(len(predictions)):

    score = chrf_score(
        predictions[i],
        references[i]
    )

    scores.append(score)

    print(f"Câu {i+1}: {score:.2f}")


# ==============================
# TÍNH chrF TOÀN BỘ DATASET
# ==============================

dataset_chrf = sum(scores) / len(scores)

print("\n===== KẾT QUẢ =====")
print(f"chrF toàn bộ dataset: {dataset_chrf:.2f}")

===== chrF TỪNG CÂU =====
Câu 1: 100.00
Câu 2: 100.00
Câu 3: 49.02
Câu 4: 100.00
Câu 5: 74.79

===== KẾT QUẢ =====
chrF toàn bộ dataset: 84.76
